# Development of the MLP architecture

## Imports

In [1]:
from protgen.data import ProteinDataset
from protgen.data.make_data import ProteinDatasetMLP
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader

## Data

In [10]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [11]:
X_train

tensor([[ 1, 13, 10,  ...,  0,  0,  0],
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13,  3,  ...,  0,  0,  0]])

In [12]:
y_train_labels 

tensor([1, 2, 1,  ..., 2, 2, 1])

In [13]:
y_train_shifted

tensor([[13, 10, 11,  ..., -1, -1, -1],
        [13, 14, 11,  ..., -1, -1, -1],
        [13, 19, 16,  ..., -1, -1, -1],
        ...,
        [13, 14, 11,  ..., -1, -1, -1],
        [13, 19, 12,  ..., -1, -1, -1],
        [13,  3,  6,  ..., -1, -1, -1]])

In [5]:
train_data.vocab

{'<PAD>': 0,
 '<BOS>': 1,
 '<EOS>': 2,
 'A': 3,
 'C': 4,
 'D': 5,
 'E': 6,
 'F': 7,
 'G': 8,
 'H': 9,
 'I': 10,
 'K': 11,
 'L': 12,
 'M': 13,
 'N': 14,
 'P': 15,
 'Q': 16,
 'R': 17,
 'S': 18,
 'T': 19,
 'V': 20,
 'W': 21,
 'Y': 22}

In [6]:
train_data.df

,sequence,family
0,MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHS...,HPr
1,MNKHELITRISEKSGMTKKDVESVLNGFMDEVTDALGSGNKVQLIG...,HU
2,MTQRFEDTFLIQNKLGLHARAASVLAELSAQFDANITLFQGSKEAE...,HPr
3,MNKTELILRIAEETGFSRKNVELGLNAAINLITKALTEEEKVQIVG...,HU
4,MVERKAVIKNEAGIHCRPTAVITQEAAKVDAVIMILSPGGTCQLGS...,HPr
...,...,...
16015,MTKADIINEIANETGVAKKDVTVVVESFMATIKDSLLEKRECVYLR...,HU
16016,MSRLSRQVTIVNQKGLHARASAKFVTFVSRLPEGLKVEVEKDGQCV...,HPr
16017,MNKSDLINHVAAETGLTKADSGFALEAVLEGITKSLRKGDTVTLTG...,HU
16018,MTLTKAELADLLFEKVGLNKREAKDMVEAFFEEIRNALERGEGVKL...,HU


In [7]:
129//4

32

In [8]:
train_data.tokens

tensor([[ 1, 13, 10,  ...,  0,  0,  0],
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13,  3,  ...,  0,  0,  0]])

In [35]:
# Constants for the dataset and model
FAMILIES = ("AcyP", "HPr", "HU")
AA = "ACDEFGHIKLMNPQRSTVWY"
special_tokens = ["<PAD>", "<BOS>", "<EOS>"]
total_tokens = list(special_tokens) + list(AA)
IGNORE_INDEX = -1  # what F.cross_entropy(ignore_index=...) must be given for the generative task

class ProteinDataset_MLP(ProteinDataset):

    def __init__(self, path_to_data):

        # Set token and family mappings
        self.vocab = dict(zip(total_tokens, range(len(total_tokens))))
        self.fam_to_idx = dict(zip(FAMILIES, range(len(FAMILIES))))
        self.window_size_mlp = 3
        self.stride_mlp = 1

        # Load raw CSV files
        self.df = pd.read_csv(path_to_data)
        self.max_length = 127 + 2    #Longest sequence + <BOS> / <EOS>

        self.tokens, self.tokens_shifted, self.labels, self.lengths = [], [], [], []
        # encode proteins and familiy
        for sequence, label in self.df[["sequence", "family"]].values:
            tokens = self.encode(sequence)
            length = len(sequence)
            shifted_tokens = self.shift_tokens(tokens, length)

            # Decided to do an additional padding to begin with so that we predict the first token
            n_pad = self.window_size_mlp - 1
            bos_pad = torch.full((n_pad,), self.vocab["<BOS>"], dtype=torch.long)
            tokens = torch.cat([bos_pad, tokens])

            # Create the MLP specfic tokens and shifted_tokens
            # I decided to do a window that has a tunabel stride.
            for index_mlp_token in range(0, length + 1 , self.stride_mlp):
                
                # Append to lists
                #[index_mlp_token:(index_mlp_token + self.window_size_mlp)
                self.tokens.append(tokens[index_mlp_token:(index_mlp_token + self.window_size_mlp)])
                self.tokens_shifted.append(shifted_tokens[index_mlp_token])
                self.labels.append(self.fam_to_idx[label])
                self.lengths.append(length)

        self.tokens = torch.stack(self.tokens)
        self.tokens_shifted = torch.stack(self.tokens_shifted)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

In [36]:
training_data_MLP =ProteinDataset_MLP(path_to_data="../data/train.csv")

print(f"Data class: {training_data_MLP}")

# Get tensors
X_train = training_data_MLP.tokens
y_train_labels = training_data_MLP.labels          # classification task
y_train_shifted = training_data_MLP.tokens_shifted # generative task

# Inddexing
x, x_shifted, y, length = training_data_MLP[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(training_data_MLP, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=1537909, vocab_size=23, num_classes=3, max_length=129)
# Sample 0: 
Tokens: 
 tensor([1, 1, 1])
 Shifted: 
 13
 Label: 1
 Length: 92


In [39]:
len(training_data_MLP)

1537909

## Test Import Data Handler

In [2]:
mlp_train = ProteinDatasetMLP(path_to_data="../data/train.csv")
print(mlp_train.tokens.shape, mlp_train.tokens_shifted.shape)  # (N, 3) and (N,)
print((mlp_train.tokens_shifted == 2).sum())                    # should be 16020: one <EOS> per protein
print(mlp_train.tokens[:3], mlp_train.tokens_shifted[:3])       # (1,1,1)→13, (1,1,13)→10, (1,13,10)→11

torch.Size([1537909, 3]) torch.Size([1537909, 1])
tensor(16020)
tensor([[ 1,  1,  1],
        [ 1,  1, 13],
        [ 1, 13, 10]]) tensor([[13],
        [10],
        [11]])


In [3]:
train_data = ProteinDatasetMLP(path_to_data="../data/train.csv")
val_data = ProteinDatasetMLP(path_to_data="../data/val.csv")
test_data = ProteinDatasetMLP(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_mlp_train = train_data.tokens
y_mlp_train_labels = train_data.labels          # classification task
y_mlp_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_mlp_train.shape}\n Labels (shape): {y_mlp_train_labels.shape} \n Shifted shape: {y_mlp_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=1537909, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([1537909, 3])
 Labels (shape): torch.Size([1537909]) 
 Shifted shape: torch.Size([1537909, 1])
# Sample 0: 
Tokens: 
 tensor([1, 1, 1])
 Shifted: 
 tensor([13])
 Label: 1
 Length: 92


In [4]:
X_mlp_train

tensor([[ 1,  1,  1],
        [ 1,  1, 13],
        [ 1, 13, 10],
        ...,
        [12, 15,  6],
        [15,  6, 19],
        [ 6, 19, 20]])

In [5]:
y_mlp_train_labels

tensor([1, 1, 1,  ..., 1, 1, 1])

In [6]:
y_mlp_train_shifted

tensor([[13],
        [10],
        [11],
        ...,
        [19],
        [20],
        [ 2]])

## Model

**Pseudo Code**

Torch stuff which I do not remember but I will check on the classes

1. Prepare our data to do this (I do not belive that we have this now) We need to break it into 3 size proteins.
    - The question that I have now is do we do this inside or outside of our loop? (I did not found any existant torch
    structure that handles this automatically)
    - Should we use all the data just consider windows of 3


2. We need to have a lookup table with the same dimension as our vocab. In each iteration we go search that one (this needs to be inside the torch loop and the lookup table needs to be optimizable). Number of dimensions in here is optimizable as well)

3. Now we concatenate them! Let's do this yee

4. Architecture to define in a standart normal neural network and the label to predict is the label at the end.
    - They suggest using tanh

5. At the end a softmax because the output should be all possible tokens

## Questions

- How do we deal with the beggining in the MLP if we want to generate from it after and we always need 3 residues to start. Is the solution something like this "like this  [ 1, 1, 1] -> 13 we should"

- What does the training looks like in the Transformer? Because in the MLP it is different correct?